In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib widget
import os
from core.helpers import get_config_file, get_data_out, get_data_computed_in
from core.preprocessing import UnscaledInputDictGenerator, AggregatedInputDictGenerator, InputVisualization, ScaledInputDictGenerator
from core.optimization import Optimization_LP
from core.postprocessing import PostProcessor

PROJECT_FILE = 'project.yaml'
CASE = 'dsm_sumo_small'
ROOT = os.path.dirname(os.path.dirname(os.getcwd()))
DATA_IN = os.path.join(ROOT, 'data', CASE, 'Input_Data')
CONFIG = get_config_file(DATA_IN, PROJECT_FILE)

DATA_COMPUTED_IN = get_data_computed_in(DATA_IN, PROJECT_FILE)
DATA_OUT         = get_data_out(DATA_IN, PROJECT_FILE)

In [ ]:
unscaled_input_dict = UnscaledInputDictGenerator(
    data_in_path = DATA_IN,
    project_file=PROJECT_FILE,
    objective= 'design',
    ignore_tsa=True
).generate_unscaled_input_dict()

aggregator = AggregatedInputDictGenerator(
    data_in_path = DATA_IN,
    project_file=PROJECT_FILE,
    unaggregated_input_dict= unscaled_input_dict
)

agg_dict = aggregator.generate_aggregated_input_dict()
aggregator.export_tsa_to_yaml()

# test LP for deterministic operation:

In [ ]:
# visualize inputs:
InputVisualization(
    data_in_path=DATA_IN, project_file=PROJECT_FILE
).visualize_unaggregated_input()

In [ ]:
# Generate unscaled_input_dict
unscaled_input_dict_generator = UnscaledInputDictGenerator(
    data_in_path = DATA_IN,
    project_file=PROJECT_FILE,
    objective= 'operation',
    name_design='ES-1'
)
unscaled_input_dict = unscaled_input_dict_generator.generate_unscaled_input_dict(
    # data_export_path=os.path.join(DATA_COMPUTED_IN, "test_unscaled")
    )

# Scale input_dict:
scaled_input_dict_generator = ScaledInputDictGenerator(
    data_in_path= DATA_IN,
    project_file= PROJECT_FILE,
    unscaled_input_dict=unscaled_input_dict
)
input_dict_scaled = scaled_input_dict_generator.generate_scaled_input_dict(
    # data_export_path=os.path.join(DATA_COMPUTED_IN, "test_scaled")
    )

# Solve LP model:
optimization_lp = Optimization_LP(
    data_in_path = DATA_IN,
    project_file = PROJECT_FILE,
    input_dict   = input_dict_scaled
)
model = optimization_lp.generate_model()
result = optimization_lp.solve_model(
    model     = model
)

# # Plot results
postprocessor = PostProcessor(
    project_file=PROJECT_FILE,
    data_in_path=DATA_IN,
    model = model,
    result= result
)

output_dict = postprocessor.build_output_dict()
postprocessor.plot_results(show_plot=True)

In [ ]:
# show values of pyomo Param:
model.production_potential_capacity.display()

In [ ]:
model.production_potential_capacity['AC'] = 3000
model.production_potential_capacity.display()

# test LP for deterministic design:

In [ ]:
# visualize inputs:
InputVisualization(
    data_in_path=DATA_IN, project_file=PROJECT_FILE
).visualize_unaggregated_input()

In [ ]:
# Generate unscaled_input_dict
unscaled_input_dict_generator = UnscaledInputDictGenerator(
    data_in_path = DATA_IN,
    project_file=PROJECT_FILE,
    objective= 'design'
)
unscaled_input_dict = unscaled_input_dict_generator.generate_unscaled_input_dict(
    # data_export_path=os.path.join(DATA_COMPUTED_IN, "test_unscaled")
    )

# Scale input_dict:
scaled_input_dict_generator = ScaledInputDictGenerator(
    data_in_path= DATA_IN,
    project_file= PROJECT_FILE,
    unscaled_input_dict=unscaled_input_dict
)
input_dict_scaled = scaled_input_dict_generator.generate_scaled_input_dict(
    # data_export_path=os.path.join(DATA_COMPUTED_IN, "test_scaled")
    )

# Solve LP model:
optimization_lp = Optimization_LP(
    data_in_path = DATA_IN,
    project_file = PROJECT_FILE,
    input_dict   = input_dict_scaled
)
model = optimization_lp.generate_model()
result = optimization_lp.solve_model(
    model     = model
)

# # Plot results
postprocessor = PostProcessor(
    project_file=PROJECT_FILE,
    data_in_path=DATA_IN,
    model = model,
    result= result
)

output_dict = postprocessor.build_output_dict()
postprocessor.plot_results(show_plot=True)

# test LP for stochastic design:
not really relevant atm

In [ ]:
# generate scenarios with three uncertian parameters:
scenario_df = ScenarioGenerator(DATA_IN, project_file=PROJECT_FILE).generate_scenarios_monte_carlo(
    n_scenarios=CONFIG['N_scenarios'],
    output_path=os.path.join(DATA_COMPUTED_IN, "test_scenarios")
)

# Generate unscaled_input_dict
unscaled_input_dict_generator = UnscaledInputDictGenerator(
    data_in_path = DATA_IN,
    project_file=PROJECT_FILE,
    objective= 'design'
)
unscaled_input_dict = unscaled_input_dict_generator.generate_unscaled_input_dict(
    # data_export_path=os.path.join(DATA_COMPUTED_IN, "test_unscaled")
    )

# Scale input_dict:
scaled_input_dict_generator = ScaledInputDictGenerator(
    data_in_path= DATA_IN,
    project_file= PROJECT_FILE,
    unscaled_input_dict=unscaled_input_dict,
    scenario_df=scenario_df
)
input_dict_scaled = scaled_input_dict_generator.generate_scaled_input_dict(
    # data_export_path=os.path.join(DATA_COMPUTED_IN, "test_scaled")
    )

# Solve LP model:
optimization_lp = Optimization_LP(
    data_in_path = DATA_IN,
    project_file = PROJECT_FILE,
    input_dict   = input_dict_scaled
)
model = optimization_lp.generate_model()
result = optimization_lp.solve_model(
    model     = model,
    mipgap    = optimization_lp.config['MIP_gap'],
    timelimit = optimization_lp.config['timelimit']
)

# Plot results
PostProcessor(
    project_file=PROJECT_FILE,
    data_in_path=DATA_IN,
    model = model,
    result= result
).plot_results(scenario_name='s00000')